# 운영 지표와 Ch18 궤적 비교

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q agentevals==0.0.9 langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 환경 준비

### 코드 읽기와 비교 경계

공통 열로 변환한다는 것은 서로 다른 기록에서 이름과 단위를 맞춰 한 표로 읽게 한다는 뜻입니다. Ch18의 `quality/5`는 0~1로 나누고, Ch21의 `agent_supported`는 불리언을 0 또는 1로 바꿉니다. 두 값이 모두 `final_score` 열에 들어가도 원래 평가 기준이 같아지는 것은 아닙니다. `trace_source`를 함께 남기는 이유는 숫자가 어느 실행과 판정식에서 왔는지 다시 확인하기 위해서입니다.

nearest-rank p95는 값을 정렬한 뒤 `ceil(0.95*n)`번째 값을 고릅니다. 표본이 세 개면 순위가 3이므로 p95는 항상 최댓값과 같습니다. 이는 계산 오류가 아니지만 꼬리 지연의 안정적인 분포를 추정하기에는 자료가 너무 적습니다. 평균과 p95를 함께 볼 때도 과제, 입력 길이, 실행 횟수, 모델 버전이 다른 집단을 직접 우열 비교하지 않습니다. 같은 서비스의 반복 실행을 일정 기간 모은 뒤 실패율과 사용자 수정 횟수도 함께 해석해야 합니다.


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
print("agentevals", version("agentevals"))
print("langchain", version("langchain"))

agentevals 0.0.9
langchain 1.4.0


## 프로젝트 루트 찾기

In [3]:
# 현재 폴더부터 상위로 탐색해 개인 절대 경로 없이 Ch18·Ch21 산출물을 찾습니다.
import json
from pathlib import Path
def find_project_root(start: Path) -> Path:
    """필요한 textbook과 labs 폴더가 함께 있는 상위 경로를 찾습니다."""
    for folder in [start, *start.parents]:
        if (folder / "textbook").is_dir() and (folder / "labs").is_dir():
            return folder
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")
root = find_project_root(Path.cwd())
print("project root:", root.name)
assert (root / "커리큘럼_v6.html").is_file()

project root: 한국경제신문


## Ch18 실제 워크플로우 출력 읽기

In [4]:
# 실행 완료된 Ch18 비교 노트북에서 다섯 패턴의 저장 출력 행을 읽습니다.
ch18_path = root / "labs/part5/ch18-workflow-patterns/notebooks/06-같은-과제로-다섯-패턴-비교.ipynb"
ch18_nb = json.loads(ch18_path.read_text(encoding="utf-8"))
printed = "\n".join("".join(out.get("text", [])) for cell in ch18_nb["cells"] for out in cell.get("outputs", []) if out.get("output_type") == "stream")
table_lines = [line for line in printed.splitlines() if " | " in line and not line.startswith("pattern")]
print("Ch18 실제 표 행:", len(table_lines))
print("\n".join(table_lines[-5:]))
assert len(table_lines) >= 5

Ch18 실제 표 행: 6
chaining | 2 | 428 | 3.513 | 5 | ok
routing | 2 | 270 | 2.33 | 4 | ok
parallel | 4 | 884 | 7.008 | 4 | ok
orchestrator_worker | 4 | 1049 | 5.11 | 5 | ok
generator_evaluator | 2 | 420 | 2.765 | 4 | ok


## Ch18 출력 공통 필드로 변환

In [5]:
# Ch18의 calls·tokens·latency·quality를 공통 운영 열로 변환합니다.
workflow_rows: list[dict] = []
for line in table_lines[-5:]:
    pattern, calls, tokens, latency, quality, status = [part.strip() for part in line.split("|")]
    workflow_rows.append({"system": f"Ch18:{pattern}", "final_score": int(quality) / 5, "model_calls": int(calls), "tool_calls": 0, "tokens": int(tokens), "latency_s": float(latency), "trace_source": "Ch18 notebook output", "status": status})
print(workflow_rows)
assert all(row["status"] == "ok" for row in workflow_rows)

[{'system': 'Ch18:chaining', 'final_score': 1.0, 'model_calls': 2, 'tool_calls': 0, 'tokens': 428, 'latency_s': 3.513, 'trace_source': 'Ch18 notebook output', 'status': 'ok'}, {'system': 'Ch18:routing', 'final_score': 0.8, 'model_calls': 2, 'tool_calls': 0, 'tokens': 270, 'latency_s': 2.33, 'trace_source': 'Ch18 notebook output', 'status': 'ok'}, {'system': 'Ch18:parallel', 'final_score': 0.8, 'model_calls': 4, 'tool_calls': 0, 'tokens': 884, 'latency_s': 7.008, 'trace_source': 'Ch18 notebook output', 'status': 'ok'}, {'system': 'Ch18:orchestrator_worker', 'final_score': 1.0, 'model_calls': 4, 'tool_calls': 0, 'tokens': 1049, 'latency_s': 5.11, 'trace_source': 'Ch18 notebook output', 'status': 'ok'}, {'system': 'Ch18:generator_evaluator', 'final_score': 0.8, 'model_calls': 2, 'tool_calls': 0, 'tokens': 420, 'latency_s': 2.765, 'trace_source': 'Ch18 notebook output', 'status': 'ok'}]


## Ch21 실제 Agentic RAG 기록 읽기

In [6]:
# Ch21이 실제 실행 후 저장한 세 질문의 답·trace·usage JSON을 읽습니다.
agent_path = root / "labs/part6/ch21-create-agent/data/rag-comparison.json"
agent_records = json.loads(agent_path.read_text(encoding="utf-8"))
print("Ch21 실제 행:", len(agent_records), "qid:", [row["qid"] for row in agent_records])
assert len(agent_records) == 3

Ch21 실제 행: 3 qid: ['C01', 'C05', 'C07']


## Agent 궤적을 공통 필드로 변환

In [7]:
# 모든 AIMessage 토큰과 tool_calls를 Ch18과 같은 열 이름으로 변환합니다.
agent_rows: list[dict] = []
for record in agent_records:
    usage = record["agent_usage"]
    agent_rows.append({"system": f"Ch21:{record['qid']}", "final_score": float(record["agent_supported"]), "model_calls": usage["model_calls"], "tool_calls": record["tool_calls"], "tokens": usage["input_tokens"] + usage["output_tokens"], "latency_s": record["agent_latency_s"], "trace_source": record["agent_trace"]})
print(agent_rows)
assert all(row["tool_calls"] >= 1 for row in agent_rows)

[{'system': 'Ch21:C01', 'final_score': 1.0, 'model_calls': 2, 'tool_calls': 1, 'tokens': 526, 'latency_s': 2.007, 'trace_source': [{'query': '책을 빌릴 수 있는 기간 며칠 대출 기간 도서 대출', 'doc_ids': ['LIB-004', 'LIB-001']}]}, {'system': 'Ch21:C05', 'final_score': 1.0, 'model_calls': 2, 'tool_calls': 1, 'tokens': 492, 'latency_s': 1.538, 'trace_source': [{'query': '토요일 자료실 폐관 시각', 'doc_ids': ['FAC-003', 'FAC-004']}]}, {'system': 'Ch21:C07', 'final_score': 1.0, 'model_calls': 2, 'tool_calls': 1, 'tokens': 541, 'latency_s': 2.094, 'trace_source': [{'query': '휴학 중 원격 전자자료 열람 가능 여부 휴학 전자자료 원격', 'doc_ids': ['ACC-002', 'ACC-004']}]}]


## 실제 두 기록 비교

In [8]:
# 서로 다른 표본임을 유지하면서 각 기록 안의 평균 품질·호출·토큰·지연을 계산합니다.
from statistics import mean
for label, rows in [("Ch18 workflow", workflow_rows), ("Ch21 agent", agent_rows)]:
    summary = {"system": label, "runs": len(rows), "final_score_mean": round(mean(row["final_score"] for row in rows), 3), "model_calls_mean": round(mean(row["model_calls"] for row in rows), 3), "tool_calls_mean": round(mean(row["tool_calls"] for row in rows), 3), "tokens_mean": round(mean(row["tokens"] for row in rows), 1), "latency_mean_s": round(mean(row["latency_s"] for row in rows), 3)}
    print(summary)
assert workflow_rows and agent_rows

{'system': 'Ch18 workflow', 'runs': 5, 'final_score_mean': 0.88, 'model_calls_mean': 2.8, 'tool_calls_mean': 0, 'tokens_mean': 610.2, 'latency_mean_s': 4.145}
{'system': 'Ch21 agent', 'runs': 3, 'final_score_mean': 1.0, 'model_calls_mean': 2, 'tool_calls_mean': 1, 'tokens_mean': 519.7, 'latency_mean_s': 1.88}


## 실제 Agent 기록의 p95 계산

In [9]:
# Ch21 세 실행의 실제 지연과 토큰을 nearest-rank p95로 계산합니다.
import math
def nearest_rank_p95(values: list[float]) -> float:
    """표본을 정렬하고 ceil(0.95*n)번째 값을 반환합니다."""
    ordered = sorted(values)
    return ordered[math.ceil(0.95 * len(ordered)) - 1]
agent_latency_p95 = nearest_rank_p95([row["latency_s"] for row in agent_rows])
agent_tokens_p95 = nearest_rank_p95([float(row["tokens"]) for row in agent_rows])
print({"actual_agent_runs": len(agent_rows), "latency_p95_s": agent_latency_p95, "tokens_p95": agent_tokens_p95})
assert agent_latency_p95 >= max(row["latency_s"] for row in agent_rows)
print("세 건의 작은 표본이므로 운영 기준을 정하려면 더 많은 실제 대화가 필요합니다.")

{'actual_agent_runs': 3, 'latency_p95_s': 2.094, 'tokens_p95': 541.0}
세 건의 작은 표본이므로 운영 기준을 정하려면 더 많은 실제 대화가 필요합니다.
